<a href="https://colab.research.google.com/github/davis-mironga/marsabit-ecosystem-analysis/blob/main/02_LULC_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — LULC Classification
## The Marsabit Footprint: Random Forest Land Cover Classification (1990–2024)

---

**Purpose of this Notebook:**
Using the NDVI composites exported in Notebook 1, we train a Random Forest
machine learning model to classify every pixel in Marsabit County into one
of 5 land cover types — for each of our 5 time periods.

**What this notebook produces:**
- ✅ Trained Random Forest classifier (per time period)
- ✅ Land cover maps for 1990, 2000, 2010, 2020, 2024
- ✅ Accuracy reports (Overall Accuracy, Kappa, Confusion Matrix)
- ✅ Change detection matrix (e.g. Forest → Bareland transitions)
- ✅ Area statistics per class per decade (hectares)
- ✅ All classified maps exported to GEE Assets

**The 5 Land Cover Classes:**
| Class | Code | Description |
|-------|------|-------------|
| Forest | 1 | Dense closed canopy — Mt. Marsabit & Mt. Kulal |
| Rangeland | 2 | Open grassland and shrubland — mid-elevation plains |
| Wetlands | 3 | Seasonally/permanently wet — riparian zones & pans |
| River | 4 | Permanent river channels — Milgis, Uaso Nyiro |
| Bareland | 5 | Exposed soil, rock, desert margins |

**Notebooks in this project:**
| Notebook | Purpose | Status |
|----------|---------|--------|
| 01_GEE_Data_Preprocessing | Satellite data, NDVI, exports | ✅ Complete |
| **02_LULC_Classification** ← You are here | Random Forest land cover maps | 🔄 In Progress |
| 03_Spatial_Analysis | Moran's I, LISA hotspots, LCI | ⏳ Pending |
| 04_Regression_Modeling | OLS, GWR, vulnerability map | ⏳ Pending |

---

## ⚙️ STEP 1 — Install Libraries, Authenticate & Load GEE Assets

**What we are doing:**
We reinstall geemap, re-authenticate GEE, and immediately load all 13 assets
that were exported in Notebook 1. Loading from GEE Assets means we do NOT
need to reprocess any satellite imagery — all the heavy lifting was already
done in Notebook 1.

**Why we reload everything:**
Colab is a temporary environment — every new session loses all variables.
By loading from GEE Assets at the start, Notebook 2 is fully independent
and can run on its own without re-running Notebook 1.

> ⚠️ When ee.Authenticate() runs, log in with the same Google account
> used in Notebook 1 (davismironga@gmail.com).

In [ ]:
# Install geemap (required every new Colab session)
!pip install geemap -q

import ee
import geemap
import math

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='mironga-project-marsabit')

# ─────────────────────────────────────────────────────────────────────
# Load all 13 assets exported from Notebook 1
# These are permanently stored in GEE — no reprocessing needed
# ─────────────────────────────────────────────────────────────────────
ASSET_PATH = 'projects/mironga-project-marsabit/assets/marsabit'

# NDVI Composites
ndvi_1990 = ee.Image(f'{ASSET_PATH}/ndvi_1990')
ndvi_2000 = ee.Image(f'{ASSET_PATH}/ndvi_2000')
ndvi_2010 = ee.Image(f'{ASSET_PATH}/ndvi_2010')
ndvi_2020 = ee.Image(f'{ASSET_PATH}/ndvi_2020')
ndvi_2024 = ee.Image(f'{ASSET_PATH}/ndvi_2024')

# NDVI Change Maps
ndvi_change_90_00 = ee.Image(f'{ASSET_PATH}/ndvi_change_1990_2000')
ndvi_change_00_10 = ee.Image(f'{ASSET_PATH}/ndvi_change_2000_2010')
ndvi_change_10_20 = ee.Image(f'{ASSET_PATH}/ndvi_change_2010_2020')
ndvi_change_20_24 = ee.Image(f'{ASSET_PATH}/ndvi_change_2020_2024')
ndvi_change_total = ee.Image(f'{ASSET_PATH}/ndvi_change_total')

# Control Variables
rainfall_change = ee.Image(f'{ASSET_PATH}/rainfall_change')
elevation       = ee.Image(f'{ASSET_PATH}/elevation')
lci_distance    = ee.Image(f'{ASSET_PATH}/lci_distance')

# Marsabit boundary
marsabit_roi  = (ee.FeatureCollection("FAO/GAUL/2015/level2")
                .filter(ee.Filter.eq('ADM2_NAME', 'Marsabit')))
marsabit_geom = marsabit_roi.geometry()

print("✅ GEE authenticated and initialized!")
print(f"\n📦 Assets loaded from: {ASSET_PATH}/")
print("   ✓ ndvi_1990, ndvi_2000, ndvi_2010, ndvi_2020, ndvi_2024")
print("   ✓ ndvi_change layers (4 decadal + 1 total)")
print("   ✓ rainfall_change, elevation, lci_distance")
print("   ✓ marsabit_roi boundary")
print("\n✅ Notebook 2 ready — no satellite reprocessing needed!")

## 🗺️ STEP 2 — Load Marsabit Boundary & Build Feature Stack

**What we are doing:**
Before we can classify land cover, we need to build a "feature stack" — a
multi-band image where each band is a different variable that helps the
Random Forest model tell land cover classes apart.

**Why one band (NDVI alone) is not enough:**
NDVI tells us how green a pixel is — but it cannot distinguish between:
- A wetland and a river (both have high NDVI near water)
- Rangeland and degraded forest (similar NDVI values)

By stacking multiple bands, we give the classifier more information to
work with — making it smarter and more accurate.

**The feature bands we stack for each time period:**
| Band | What it measures | Why it helps |
|------|-----------------|--------------|
| NDVI | Vegetation greenness | Primary vegetation signal |
| NIR | Near-infrared reflectance | Separates vegetation types |
| Red | Red reflectance | Separates soil from vegetation |
| SWIR | Shortwave infrared | Separates wetlands from dryland |
| Elevation | Terrain height | Forest only exists at high elevation |
| Slope | Terrain steepness | Affects grazing accessibility |

**Output of this step:**
A feature stack image for each of the 5 time periods — each with 6 bands.
The Random Forest classifier will train and predict on these stacks.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# We need the original Landsat composites to extract NIR, Red, SWIR
# These raw band values give the classifier more to work with than
# NDVI alone. We reload them from the original GEE collections.
# ─────────────────────────────────────────────────────────────────────

def apply_scale_factors(image):
    """Apply Landsat C2 L2 scale factors to optical bands."""
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using QA_PIXEL band."""
    qa = image.select('QA_PIXEL')
    cloud_shadow = qa.bitwiseAnd(1 << 3).eq(0)
    cloud        = qa.bitwiseAnd(1 << 5).eq(0)
    return image.updateMask(cloud_shadow.And(cloud))

def build_feature_stack(collection_id, start_year, end_year,
                        roi, sensor='L57'):
    """
    Builds a multi-band feature stack for classification.
    Bands: NDVI, NIR, Red, SWIR, Elevation, Slope

    Args:
        collection_id : GEE Landsat collection string
        start_year    : Start year as string
        end_year      : End year as string
        roi           : GEE geometry
        sensor        : 'L57' for Landsat 5/7 | 'L89' for Landsat 8/9

    Returns:
        6-band image clipped to roi
    """
    collection = (
        ee.ImageCollection(collection_id)
        .filterBounds(roi)
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        .map(mask_landsat_clouds)
        .map(apply_scale_factors)
    )

    composite = collection.median().clip(roi)

    # Select bands based on sensor generation
    if sensor == 'L57':
        nir_b, red_b, swir_b = 'SR_B4', 'SR_B3', 'SR_B5'
    else:
        nir_b, red_b, swir_b = 'SR_B5', 'SR_B4', 'SR_B6'

    nir  = composite.select(nir_b).rename('NIR')
    red  = composite.select(red_b).rename('Red')
    swir = composite.select(swir_b).rename('SWIR')
    ndvi = composite.normalizedDifference([nir_b, red_b]).rename('NDVI')

    # Elevation and slope (same for all time periods)
    srtm  = ee.Image("USGS/SRTMGL1_003")
    elev  = srtm.select('elevation').rename('Elevation')
    slope = ee.Terrain.slope(srtm).rename('Slope')

    # Stack all bands into one image
    stack = (ndvi
             .addBands(nir)
             .addBands(red)
             .addBands(swir)
             .addBands(elev)
             .addBands(slope)
             .clip(roi))

    return stack


# ─────────────────────────────────────────────────────────────────────
# Build feature stacks for all 5 time periods
# ─────────────────────────────────────────────────────────────────────
print("⏳ Building feature stacks for all 5 periods...")

stack_1990 = build_feature_stack(
    'LANDSAT/LT05/C02/T1_L2', '1988', '1992', marsabit_geom, 'L57')
print("   ✓ Feature stack 1990 (bands: NDVI, NIR, Red, SWIR, Elevation, Slope)")

stack_2000 = build_feature_stack(
    'LANDSAT/LT05/C02/T1_L2', '1998', '2002', marsabit_geom, 'L57')
print("   ✓ Feature stack 2000")

stack_2010 = build_feature_stack(
    'LANDSAT/LE07/C02/T1_L2', '2008', '2012', marsabit_geom, 'L57')
print("   ✓ Feature stack 2010")

stack_2020 = build_feature_stack(
    'LANDSAT/LC08/C02/T1_L2', '2018', '2022', marsabit_geom, 'L89')
print("   ✓ Feature stack 2020")

stack_2024 = build_feature_stack(
    'LANDSAT/LC09/C02/T1_L2', '2022', '2024', marsabit_geom, 'L89')
print("   ✓ Feature stack 2024")

# Visualize the 2020 stack as a true colour composite to confirm
rgb_vis = {'bands': ['NIR', 'Red', 'SWIR'], 'min': 0, 'max': 0.4}

Map = geemap.Map()
Map.setCenter(37.9, 2.3, 8)
Map.addLayer(stack_2020, rgb_vis, 'False Colour Composite 2020')
Map.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map.addLayerControl()

print("\n✅ All 5 feature stacks ready!")
print("   Each stack has 6 bands: NDVI, NIR, Red, SWIR, Elevation, Slope")
print("   Map shows 2020 false colour composite — vegetation appears red/pink")
Map

## 🎯 STEP 3 — Create Training Samples (Reference Points per Class)

**What we are doing:**
We create training samples — a set of known locations where we are CERTAIN
about the land cover type. These known points teach the Random Forest model
what each class looks like in satellite data.

**How Random Forest classification works (simply):**
1. We show the model 100s of examples: "this pixel = Forest", "this pixel = Bareland"
2. The model learns the spectral signature of each class
3. We then ask it to classify EVERY pixel in Marsabit
4. It predicts the most likely class for each pixel based on what it learned

**Why we use geometry points defined in code (not manual digitizing):**
We define training points using known geographic coordinates of landmark
locations in Marsabit — places we know with certainty what the land cover is:
- Mt. Marsabit Forest Reserve → Forest
- Chalbi Desert → Bareland  
- Lake Turkana shore → Wetlands
- Uaso Nyiro River → River
- Open plains around Laisamis → Rangeland

**Training sample design:**
| Class | Label Code | No. of Points | Known Locations |
|-------|-----------|---------------|-----------------|
| Forest | 1 | 50 | Mt. Marsabit, Mt. Kulal |
| Rangeland | 2 | 50 | Laisamis plains, North Horr |
| Wetlands | 3 | 30 | Lake Turkana shores, seasonal pans |
| River | 4 | 30 | Milgis Lugga, Uaso Nyiro |
| Bareland | 5 | 50 | Chalbi Desert, Dida Galgalu |

> 💡 We use 70% of these points for TRAINING the model and
> 30% for TESTING (validating accuracy) — this is the 70/30 split
> specified in our project documentation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# STEP 3 (MEMORY OPTIMISED) — Reduced samples + coarser sampling scale
# GEE memory limit fix: sample at 90m instead of 30m
# Fewer points but still enough for reliable accuracy assessment
# ─────────────────────────────────────────────────────────────────────

FEATURE_BANDS = ['NDVI', 'NIR', 'Red', 'SWIR', 'Elevation', 'Slope']
LABEL         = 'landcover'
N_SAMPLES     = 50   # 50 per class = 250 total → ~175 train, ~75 test
SAMPLE_SCALE  = 90   # 90m sampling scale to stay within GEE memory limits

reference = stack_2020

# ── Spectral masks for each class ────────────────────────────────────
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence')

forest_mask   = (reference.select('NDVI').gt(0.5)
                 .And(reference.select('Elevation').gt(1200)))

rangeland_mask = (reference.select('NDVI').gt(0.15)
                  .And(reference.select('NDVI').lt(0.45))
                  .And(reference.select('Elevation').lt(1200)))

wetland_mask  = (jrc.gt(10).And(jrc.lt(50))
                 .And(reference.select('NDVI').gt(0.1)))

river_mask    = jrc.gte(50)

bareland_mask = (reference.select('NDVI').lt(0.12)
                 .And(reference.select('Elevation').lt(800)))

# ─────────────────────────────────────────────────────────────────────
# Sample each class separately with tileScale=4 to reduce memory
# tileScale divides computation into smaller tiles
# ─────────────────────────────────────────────────────────────────────
def sample_class(mask, class_code, n, roi):
    """
    Sample n points from a binary mask.
    Uses coarser scale and tileScale to avoid memory errors.
    """
    labeled = mask.selfMask().rename('landcover').toByte()
    feature_image = reference.select(FEATURE_BANDS)

    samples = (labeled
               .addBands(feature_image)
               .stratifiedSample(
                   numPoints=n,
                   classBand='landcover',
                   region=roi,
                   scale=SAMPLE_SCALE,
                   tileScale=4,       # ← reduces memory usage
                   seed=42,
                   geometries=True
               )
               .map(lambda f: f.set('landcover', class_code)))
    return samples

print("⏳ Auto-sampling training points (memory-optimised)...")

forest_samples    = sample_class(forest_mask,    1, N_SAMPLES, marsabit_geom)
print(f"   ✓ Forest    : {forest_samples.size().getInfo()} points")

rangeland_samples = sample_class(rangeland_mask, 2, N_SAMPLES, marsabit_geom)
print(f"   ✓ Rangeland : {rangeland_samples.size().getInfo()} points")

wetland_samples   = sample_class(wetland_mask,   3, N_SAMPLES, marsabit_geom)
print(f"   ✓ Wetlands  : {wetland_samples.size().getInfo()} points")

river_samples     = sample_class(river_mask,     4, N_SAMPLES, marsabit_geom)
print(f"   ✓ River     : {river_samples.size().getInfo()} points")

bareland_samples  = sample_class(bareland_mask,  5, N_SAMPLES, marsabit_geom)
print(f"   ✓ Bareland  : {bareland_samples.size().getInfo()} points")

# ─────────────────────────────────────────────────────────────────────
# Merge all classes
# ─────────────────────────────────────────────────────────────────────
all_training_points = (forest_samples
                       .merge(rangeland_samples)
                       .merge(wetland_samples)
                       .merge(river_samples)
                       .merge(bareland_samples))

total = all_training_points.size().getInfo()
print(f"\n✅ Total training points : {total}")
print(f"   Estimated train (70%) : ~{int(total*0.7)}")
print(f"   Estimated test  (30%) : ~{int(total*0.3)}")

# ─────────────────────────────────────────────────────────────────────
# Visualize on map
# ─────────────────────────────────────────────────────────────────────
Map_s = geemap.Map()
Map_s.setCenter(37.9, 2.3, 8)
Map_s.addLayer(forest_samples,    {'color': '1a7a4a'}, 'Forest')
Map_s.addLayer(rangeland_samples, {'color': 'a6d96a'}, 'Rangeland')
Map_s.addLayer(wetland_samples,   {'color': '4575b4'}, 'Wetlands')
Map_s.addLayer(river_samples,     {'color': '74add1'}, 'River')
Map_s.addLayer(bareland_samples,  {'color': 'd73027'}, 'Bareland')
Map_s.addLayer(marsabit_roi,      {'color': 'white'},  'Marsabit Boundary')
Map_s.addLayerControl()

print("\n💡 Map shows auto-sampled points spread across Marsabit")
print("   Green=Forest | Yellow=Rangeland | Dark Blue=Wetlands")
print("   Light Blue=River | Red=Bareland")
Map_s

## 🔬 STEP 4 — Extract Spectral Features for Training

**What we are doing:**
We sample the feature stack values (NDVI, NIR, Red, SWIR, Elevation, Slope)
at each of our training point locations. This creates a training table where:
- Each ROW = one training point location
- Each COLUMN = one spectral/terrain feature value at that location
- The LABEL column = the known land cover class (1–5)

**Why this step is necessary:**
The Random Forest model does not learn from coordinates — it learns from
numbers. We need to extract the actual pixel values from the satellite image
at each training point location.

**The training table structure:**
| Point | NDVI | NIR | Red | SWIR | Elevation | Slope | Class |
|-------|------|-----|-----|------|-----------|-------|-------|
| 1 | 0.72 | 0.38 | 0.06 | 0.18 | 1680 | 12.3 | 1 (Forest) |
| 2 | 0.08 | 0.14 | 0.12 | 0.22 | 380 | 1.2 | 5 (Bareland) |
| ... | ... | ... | ... | ... | ... | ... | ... |

**70/30 Train-Test Split:**
We randomly split all sampled points into:
- 70% Training set → the model LEARNS from these
- 30% Testing set  → we use these to MEASURE accuracy (model never sees them during training)

This split prevents overfitting — the model cannot just memorise the answers.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Extract feature values at all training point locations
# We use the 2020 feature stack as our primary training source
# (most recent cloud-free data with best sensor quality)
# ─────────────────────────────────────────────────────────────────────

# Define the band names the classifier will use
FEATURE_BANDS = ['NDVI', 'NIR', 'Red', 'SWIR', 'Elevation', 'Slope']
LABEL         = 'landcover'

# Sample the feature stack at each training point
training_samples = stack_2020.select(FEATURE_BANDS).sampleRegions(
    collection=all_training_points,
    properties=[LABEL],
    scale=30,          # 30m — matches Landsat resolution
    geometries=True    # Keep geometry for visualization
)

# ─────────────────────────────────────────────────────────────────────
# 70/30 Train-Test Split
# We add a random column then split on its value
# ─────────────────────────────────────────────────────────────────────
training_samples = training_samples.randomColumn('random', seed=42)

train_set = training_samples.filter(ee.Filter.lte('random', 0.7))
test_set  = training_samples.filter(ee.Filter.gt('random', 0.7))

# Report split sizes
total_sampled = training_samples.size().getInfo()
train_size    = train_set.size().getInfo()
test_size     = test_set.size().getInfo()

print("✅ Feature extraction complete!")
print(f"   Total sampled points : {total_sampled}")
print(f"   Training set (70%)   : {train_size} points")
print(f"   Testing set  (30%)   : {test_size} points")
print(f"\n   Feature bands used   : {FEATURE_BANDS}")
print(f"   Label column         : '{LABEL}' (1=Forest, 2=Rangeland,")
print(f"                           3=Wetlands, 4=River, 5=Bareland)")
print("\n✅ 70/30 split applied — model ready for training!")

## 🌲 STEP 5 — Train the Random Forest Classifier

**What we are doing:**
We train a Random Forest classifier using the training set from Step 4.
The trained model is then applied to classify ALL pixels across Marsabit
for all 5 time periods.

**What is Random Forest? (Plain English)**
Imagine asking 100 different experts to look at a pixel and vote on what
land cover type it is. Each expert (decision tree) looks at a random
subset of the features (NDVI, NIR, elevation etc.) and casts a vote.
The final classification = the class that gets the most votes.

This "wisdom of crowds" approach makes Random Forest:
- More accurate than a single decision tree
- Resistant to overfitting
- Able to handle complex non-linear relationships

**Our Random Forest settings:**
| Parameter | Value | Meaning |
|-----------|-------|---------|
| numberOfTrees | 100 | 100 decision trees vote on each pixel |
| seed | 42 | Ensures reproducible results |
| Features | 6 bands | NDVI, NIR, Red, SWIR, Elevation, Slope |
| Classes | 5 | Forest, Rangeland, Wetlands, River, Bareland |

**Target accuracy: ≥ 85% Overall Accuracy**
This is the minimum standard specified in our project documentation.

**What the classified map will show:**
Every single pixel in Marsabit assigned to one of 5 colour-coded classes.
We classify all 5 time periods using the same trained model.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# TRAIN the Random Forest Classifier
# ─────────────────────────────────────────────────────────────────────
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42
).train(
    features=train_set,
    classProperty=LABEL,
    inputProperties=FEATURE_BANDS
)

print("✅ Random Forest classifier trained!")
print("   → numberOfTrees : 100")
print("   → Features      : NDVI, NIR, Red, SWIR, Elevation, Slope")
print("   → Classes       : 5 (Forest, Rangeland, Wetlands, River, Bareland)")

# ─────────────────────────────────────────────────────────────────────
# CLASSIFY all 5 time periods using the trained model
# ─────────────────────────────────────────────────────────────────────
print("\n⏳ Classifying all 5 time periods...")

lulc_1990 = (stack_1990.select(FEATURE_BANDS)
             .classify(classifier)
             .rename('landcover')
             .clip(marsabit_geom))
print("   ✓ 1990 classified")

lulc_2000 = (stack_2000.select(FEATURE_BANDS)
             .classify(classifier)
             .rename('landcover')
             .clip(marsabit_geom))
print("   ✓ 2000 classified")

lulc_2010 = (stack_2010.select(FEATURE_BANDS)
             .classify(classifier)
             .rename('landcover')
             .clip(marsabit_geom))
print("   ✓ 2010 classified")

lulc_2020 = (stack_2020.select(FEATURE_BANDS)
             .classify(classifier)
             .rename('landcover')
             .clip(marsabit_geom))
print("   ✓ 2020 classified")

lulc_2024 = (stack_2024.select(FEATURE_BANDS)
             .classify(classifier)
             .rename('landcover')
             .clip(marsabit_geom))
print("   ✓ 2024 classified")

# ─────────────────────────────────────────────────────────────────────
# Visualization palette — one colour per class
# ─────────────────────────────────────────────────────────────────────
lulc_vis = {
    'min': 1,
    'max': 5,
    'palette': [
        '1a7a4a',   # Class 1 = Forest      → Dark Green
        'a6d96a',   # Class 2 = Rangeland   → Light Green
        '4575b4',   # Class 3 = Wetlands    → Blue
        '74add1',   # Class 4 = River       → Light Blue
        'd73027'    # Class 5 = Bareland    → Red
    ]
}

# ─────────────────────────────────────────────────────────────────────
# Visualize on interactive map — toggle years to see change
# ─────────────────────────────────────────────────────────────────────
Map3 = geemap.Map()
Map3.setCenter(37.9, 2.3, 8)
Map3.addLayer(lulc_1990, lulc_vis, 'LULC 1990')
Map3.addLayer(lulc_2000, lulc_vis, 'LULC 2000', False)
Map3.addLayer(lulc_2010, lulc_vis, 'LULC 2010', False)
Map3.addLayer(lulc_2020, lulc_vis, 'LULC 2020', False)
Map3.addLayer(lulc_2024, lulc_vis, 'LULC 2024')
Map3.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map3.addLayerControl()

print("\n✅ All 5 time periods classified successfully!")
print("─────────────────────────────────────────────")
print("   🟢 Dark Green  = Forest")
print("   🟡 Light Green = Rangeland")
print("   🔵 Dark Blue   = Wetlands")
print("   🩵 Light Blue  = River")
print("   🔴 Red         = Bareland")
print("─────────────────────────────────────────────")
print("💡 Toggle 1990 vs 2024 on the map to see 36 years of change!")
Map3

## 📊 STEP 6 — Accuracy Assessment (Confusion Matrix, Kappa, Overall Accuracy)

**What we are doing:**
We measure how accurate our Random Forest classification is by testing it
against the 30% of points it never saw during training (the test set).

**Why accuracy assessment matters:**
Without this step, we have no idea if our classified map is 85% accurate
or 50% accurate. A map that looks good visually can still be scientifically
wrong. Our project documentation requires Overall Accuracy ≥ 85%.

**The three accuracy metrics we report:**

1. Overall Accuracy (OA)
   The percentage of all test points correctly classified.
   Example: 89% OA means 89 out of every 100 pixels were correctly labelled.

2. Kappa Coefficient (κ)
   Measures accuracy adjusted for chance agreement.
   | Kappa Value | Interpretation |
   |-------------|----------------|
   | > 0.80 | Strong agreement — excellent classification |
   | 0.60–0.80 | Moderate agreement — acceptable |
   | < 0.60 | Weak agreement — needs improvement |

3. Confusion Matrix
   A table showing exactly where the model makes mistakes.
   Rows = actual class | Columns = predicted class
   Diagonal = correctly classified pixels
   Off-diagonal = errors (e.g. Rangeland misclassified as Bareland)

**Producer's and User's Accuracy:**
- Producer's Accuracy = "Of all actual Forest pixels, how many did we
  correctly identify?" (avoids missing a class)
- User's Accuracy = "Of all pixels we called Forest, how many actually
  are Forest?" (avoids false alarms)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Accuracy Assessment using the 30% test set
# The classifier predicts classes for test points
# We compare predictions vs known labels
# ─────────────────────────────────────────────────────────────────────

classified_test  = test_set.classify(classifier)
conf_matrix      = classified_test.errorMatrix(
    actual='landcover',
    predicted='classification'
)

overall_accuracy = conf_matrix.accuracy().getInfo()
kappa            = conf_matrix.kappa().getInfo()
matrix_values    = conf_matrix.getInfo()

print("=" * 55)
print("   RANDOM FOREST ACCURACY ASSESSMENT REPORT")
print("=" * 55)
print(f"\n   Overall Accuracy  : {overall_accuracy * 100:.2f}%")
print(f"   Kappa Coefficient : {kappa:.4f}")

if kappa > 0.80:
    kappa_label = "Strong agreement ✅"
elif kappa > 0.60:
    kappa_label = "Moderate agreement ⚠️"
else:
    kappa_label = "Weak agreement ❌"
print(f"   Kappa Assessment  : {kappa_label}")

if overall_accuracy >= 0.85:
    print(f"\n   ✅ Target accuracy ≥ 85% ACHIEVED!")
else:
    print(f"\n   ⚠️  Below 85% target — consider adding more training points")

# ─────────────────────────────────────────────────────────────────────
# Confusion Matrix — safe printing
# GEE sometimes returns an extra empty row — we filter those out
# ─────────────────────────────────────────────────────────────────────
class_names = ['Forest', 'Rangeland', 'Wetlands', 'River', 'Bareland']

print("\n── Confusion Matrix ──────────────────────────────────")
print("   Rows = Actual Class | Columns = Predicted Class")
print("   1=Forest 2=Rangeland 3=Wetlands 4=River 5=Bareland\n")

matrix = conf_matrix.getInfo()

# Filter out empty rows (all zeros) — fixes GEE extra row issue
data_rows  = [row for row in matrix if any(v != 0 for v in row)]
n_cols     = max(len(row) for row in data_rows) if data_rows else 5
col_labels = class_names[:n_cols]

# Print header
header = f"   {'Actual/Pred':<12}" + "".join([f"{c[:9]:>11}" for c in col_labels])
print(header)
print("   " + "─" * (12 + 11 * n_cols))

# Print matrix rows
for i, row in enumerate(data_rows):
    label   = class_names[i] if i < len(class_names) else f"Class {i+1}"
    row_str = f"   {label:<12}" + "".join([f"{int(v):>11}" for v in row])
    print(row_str)

# ─────────────────────────────────────────────────────────────────────
# Per-Class Accuracy — Producer's and User's
# ─────────────────────────────────────────────────────────────────────
print("\n── Per-Class Accuracy ────────────────────────────────")
try:
    producers   = conf_matrix.producersAccuracy().getInfo()
    users       = conf_matrix.consumersAccuracy().getInfo()

    # Filter empty rows
    prod_values = [row for row in producers if any(v != 0 for v in row)]
    user_values = [v for v in users[0] if v != 0]

    print(f"   {'Class':<12} {'Producer Acc':>14} {'User Acc':>10}")
    print("   " + "─" * 40)
    for i, name in enumerate(class_names):
        pa = prod_values[i][0] * 100 if i < len(prod_values) else 0
        ua = user_values[i] * 100    if i < len(user_values) else 0
        flag = "✅" if pa >= 85 and ua >= 85 else "⚠️"
        print(f"   {name:<12} {pa:>13.1f}% {ua:>9.1f}%  {flag}")

except Exception as e:
    print(f"   Per-class accuracy unavailable: {e}")

# ─────────────────────────────────────────────────────────────────────
# Feature Importance — which band contributed most?
# ─────────────────────────────────────────────────────────────────────
print("\n── Feature Importance ────────────────────────────────")
try:
    importance     = classifier.explain().getInfo()
    importance_dict = importance.get('importance', {})
    sorted_bands   = sorted(
        importance_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )
    print(f"   {'Band':<12} {'Importance':>12}")
    print("   " + "─" * 26)
    for band, score in sorted_bands:
        bar = "█" * int(score * 30)
        print(f"   {band:<12} {score:>10.4f}  {bar}")
except Exception as e:
    print(f"   Feature importance unavailable: {e}")

print("\n" + "=" * 55)
print("✅ Accuracy Assessment Complete!")
print(f"   Overall Accuracy : {overall_accuracy * 100:.2f}%  (Target ≥ 85%)")
print(f"   Kappa            : {kappa:.4f}  (Target > 0.80)")
print("=" * 55)

## 🔄 STEP 7 — Change Detection (Area Statistics & Transition Matrix)

**What we are doing:**
Now that we have classified land cover maps for all 5 time periods, we
calculate two things:

1. Area Statistics — How many hectares of each class per decade?
   This tells us if forest is shrinking or bareland is expanding over time.

2. Transition Matrix (1990 → 2024) — What did each class BECOME?
   Example: How many hectares of Forest became Bareland over 36 years?

**How to read the Transition Matrix:**
- ROWS = what the land cover WAS in 1990
- COLUMNS = what the land cover BECAME in 2024
- DIAGONAL = land that STAYED the same class (stable)
- OFF-DIAGONAL = land that CHANGED class (transitions)

**The most policy-relevant transitions:**
| Transition | Meaning |
|------------|---------|
| Forest → Bareland | Severe irreversible degradation |
| Forest → Rangeland | Forest thinning — early degradation |
| Rangeland → Bareland | Rangeland degradation — desertification |
| Bareland → Rangeland | Recovery — vegetation returning |

**Output feeds directly into Notebook 3:**
The transition areas become the basis for identifying degradation
hotspots in the Moran's I and LISA analysis.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# AREA STATISTICS — hectares per class per decade
# 1 pixel = 30m x 30m = 900m² = 0.09 hectares
# ─────────────────────────────────────────────────────────────────────

PIXEL_AREA_HA = 900 / 10000   # 0.09 ha per 30m pixel
class_names   = ['Forest', 'Rangeland', 'Wetlands', 'River', 'Bareland']
class_codes   = [1, 2, 3, 4, 5]

def get_area_stats(lulc_image, roi):
    """Calculate area in hectares for each land cover class."""
    results = {}
    for code, name in zip(class_codes, class_names):
        mask  = lulc_image.eq(code)
        count = mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=90,          # 90m scale — matches our sampling scale
            maxPixels=1e9,
            tileScale=4
        ).getInfo()
        pixel_count    = list(count.values())[0] if count else 0
        results[name]  = round((pixel_count or 0) * PIXEL_AREA_HA, 0)
    return results

print("⏳ Computing area statistics for all 5 periods...")
print("   (This may take 2–3 minutes)\n")

areas = {}
for year, lulc in zip(
    ['1990', '2000', '2010', '2020', '2024'],
    [lulc_1990, lulc_2000, lulc_2010, lulc_2020, lulc_2024]
):
    areas[year] = get_area_stats(lulc, marsabit_geom)
    print(f"   ✓ {year} area statistics computed")

# ─────────────────────────────────────────────────────────────────────
# Print Area Statistics Table
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("   LAND COVER AREA STATISTICS (Hectares)")
print("=" * 72)
print(f"   {'Class':<12}" + "".join([f"{y:>12}" for y in areas.keys()])
      + f"{'Net Change':>14}")
print("   " + "─" * 72)

for name in class_names:
    row    = f"   {name:<12}"
    for year in areas.keys():
        ha  = areas[year].get(name, 0)
        row += f"{ha:>12,.0f}"
    change  = areas['2024'].get(name, 0) - areas['1990'].get(name, 0)
    symbol  = "▲" if change > 0 else "▼"
    row    += f"  {symbol}{abs(change):>10,.0f}"
    print(row)

print("=" * 72)

# ─────────────────────────────────────────────────────────────────────
# TRANSITION MATRIX — 1990 to 2024
# Encodes: from_class * 10 + to_class
# ─────────────────────────────────────────────────────────────────────
print("\n⏳ Computing transition matrix 1990 → 2024...")
print("   (This may take 2–3 minutes)\n")

transition_image = lulc_1990.multiply(10).add(lulc_2024).rename('transition')

# Build full matrix
transition_matrix = {}
for from_code, from_name in zip(class_codes, class_names):
    transition_matrix[from_name] = {}
    for to_code, to_name in zip(class_codes, class_names):
        code  = from_code * 10 + to_code
        mask  = transition_image.eq(code)
        count = mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=marsabit_geom,
            scale=90,
            maxPixels=1e9,
            tileScale=4
        ).getInfo()
        pixel_count = list(count.values())[0] if count else 0
        transition_matrix[from_name][to_name] = round(
            (pixel_count or 0) * PIXEL_AREA_HA, 0
        )

# Print Transition Matrix
print("=" * 72)
print("   LAND COVER TRANSITION MATRIX 1990 → 2024 (Hectares)")
print("   Rows = Land cover in 1990 | Columns = Land cover in 2024")
print("=" * 72)
print(f"   {'From / To':<12}" + "".join([f"{c[:10]:>12}" for c in class_names]))
print("   " + "─" * 72)

for from_name in class_names:
    row_str = f"   {from_name:<12}"
    for to_name in class_names:
        ha = transition_matrix[from_name].get(to_name, 0)
        # Highlight diagonal (stable) in the output
        marker = "*" if from_name == to_name else " "
        row_str += f"{ha:>11,.0f}{marker}"
    print(row_str)

print("=" * 72)
print("   * = Stable (no change) | Other values = Transitions")

# ─────────────────────────────────────────────────────────────────────
# Key Transitions Summary — policy-relevant findings
# ─────────────────────────────────────────────────────────────────────
print("\n── Key Transitions 1990 → 2024 ──────────────────────")
key_transitions = [
    ('Forest',    'Bareland',   'Severe degradation ⚠️'),
    ('Forest',    'Rangeland',  'Forest thinning ⚠️'),
    ('Rangeland', 'Bareland',   'Desertification ⚠️'),
    ('Bareland',  'Rangeland',  'Recovery ✅'),
    ('Rangeland', 'Forest',     'Forest recovery ✅'),
]

for from_name, to_name, label in key_transitions:
    ha = transition_matrix.get(from_name, {}).get(to_name, 0)
    print(f"   {from_name:<12} → {to_name:<12} : {ha:>10,.0f} ha  {label}")

print("\n✅ Change detection complete!")

## 📤 STEP 8 — Export Classified Maps to GEE Assets

**What we are doing:**
We export all 5 classified land cover maps to GEE Assets so they are
permanently stored and available for Notebook 3 (Spatial Analysis).

**What gets exported:**
| Asset Name | Description | Used In |
|------------|-------------|---------|
| lulc_1990 | Classified land cover 1990 | Notebook 3 |
| lulc_2000 | Classified land cover 2000 | Notebook 3 |
| lulc_2010 | Classified land cover 2010 | Notebook 3 |
| lulc_2020 | Classified land cover 2020 | Notebook 3 |
| lulc_2024 | Classified land cover 2024 | Notebook 3 |

**After export — total assets in GEE:**
- 13 assets from Notebook 1 (NDVI, rainfall, elevation, LCI)
- 5 assets from Notebook 2 (LULC maps)
- Total: 18 assets

> ⚠️ Monitor exports at:
> console.cloud.google.com/earth-engine/tasks?project=mironga-project-marsabit
> All tasks must show SUCCEEDED before starting Notebook 3.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Export all 5 classified LULC maps to GEE Assets
# toByte() saves as integer (class codes 1-5) — smaller file size
# ─────────────────────────────────────────────────────────────────────
ASSET_PATH = 'projects/mironga-project-marsabit/assets/marsabit'

lulc_exports = {
    'lulc_1990': lulc_1990,
    'lulc_2000': lulc_2000,
    'lulc_2010': lulc_2010,
    'lulc_2020': lulc_2020,
    'lulc_2024': lulc_2024
}

print("⏳ Submitting LULC export tasks to GEE Assets...")
print(f"   Path: {ASSET_PATH}/\n")

for asset_name, image in lulc_exports.items():
    try:
        task = ee.batch.Export.image.toAsset(
            image=image.toByte(),
            description=f'LULC_{asset_name}',
            assetId=f'{ASSET_PATH}/{asset_name}',
            region=marsabit_geom,
            scale=30,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        task.start()
        print(f"   ✅ Task submitted: {asset_name}")
    except Exception as e:
        print(f"   ❌ Error submitting {asset_name}: {e}")

print(f"\n✅ All export tasks submitted!")
print("─────────────────────────────────────────────────────")
print("📌 Monitor progress:")
print("   console.cloud.google.com/earth-engine/tasks")
print(f"   Project: mironga-project-marsabit")
print("─────────────────────────────────────────────────────")

# ─────────────────────────────────────────────────────────────────────
# Verify all assets after exports complete
# Re-run this block after tasks show SUCCEEDED in GEE
# ─────────────────────────────────────────────────────────────────────
print("\n⏳ Verifying all assets (Notebook 1 + Notebook 2)...")

expected_all = [
    # Notebook 1 assets
    'ndvi_1990', 'ndvi_2000', 'ndvi_2010', 'ndvi_2020', 'ndvi_2024',
    'ndvi_change_1990_2000', 'ndvi_change_2000_2010',
    'ndvi_change_2010_2020', 'ndvi_change_2020_2024',
    'ndvi_change_total', 'rainfall_change',
    'elevation', 'lci_distance',
    # Notebook 2 assets
    'lulc_1990', 'lulc_2000', 'lulc_2010', 'lulc_2020', 'lulc_2024'
]

assets = ee.data.listAssets({'parent': ASSET_PATH})
found  = [a['name'].split('/')[-1] for a in assets.get('assets', [])]
total_found = sum(1 for e in expected_all if e in found)

print(f"\n📦 Total Assets: {total_found}/18")
print("─────────────────────────────────────────────────────")
print("   NOTEBOOK 1 ASSETS (13):")
for name in expected_all[:13]:
    status = "✅" if name in found else "⏳ pending"
    print(f"   {status}  {name}")

print("\n   NOTEBOOK 2 ASSETS (5):")
for name in expected_all[13:]:
    status = "✅" if name in found else "⏳ pending"
    print(f"   {status}  {name}")

print("─────────────────────────────────────────────────────")
if total_found == 18:
    print("🎉 All 18 assets verified!")
    print("   Ready to proceed to Notebook 3: Spatial Analysis")
else:
    print(f"⏳ {18 - total_found} assets still pending")
    print("   Re-run this cell after GEE tasks complete")

## ✅ NOTEBOOK 2 COMPLETE — Summary

All land cover classification and change detection steps completed successfully.

---

**Classification Performance:**
| Metric | Result | Target | Status |
|--------|--------|--------|--------|
| Overall Accuracy | 97.14% | ≥ 85% | ✅ Exceeded |
| Kappa Coefficient | 0.9641 | > 0.80 | ✅ Exceeded |
| Forest Accuracy | 100.0% | ≥ 85% | ✅ Exceeded |
| Rangeland Accuracy | 100.0% | ≥ 85% | ✅ Exceeded |
| Wetlands Accuracy | 91.7% | ≥ 85% | ✅ Exceeded |
| River Accuracy | 100.0% | ≥ 85% | ✅ Exceeded |
| Bareland Accuracy | 93.8% | ≥ 85% | ✅ Exceeded |

---

**Feature Importance Ranking:**
| Rank | Band | Importance | Interpretation |
|------|------|------------|----------------|
| 1 | NDVI | 82.98 | Primary vegetation signal |
| 2 | Elevation | 80.03 | Terrain drives vegetation type |
| 3 | SWIR | 58.97 | Separates wetlands from dryland |
| 4 | Red | 53.95 | Separates soil from vegetation |
| 5 | NIR | 49.22 | Supports vegetation discrimination |
| 6 | Slope | 42.99 | Grazing accessibility indicator |

---

**Steps Completed:**
| Step | Task | Status |
|------|------|--------|
| Step 1 | Load GEE Assets from Notebook 1 | ✅ Done |
| Step 2 | Build 6-band Feature Stacks | ✅ Done |
| Step 3 | Auto-sample Training Points (5 classes) | ✅ Done |
| Step 4 | 70/30 Train-Test Split | ✅ Done |
| Step 5 | Train & Apply Random Forest Classifier | ✅ Done |
| Step 6 | Accuracy Assessment (OA, Kappa, Confusion Matrix) | ✅ Done |
| Step 7 | Change Detection (Area Stats + Transition Matrix) | ✅ Done |
| Step 8 | Export 5 LULC Maps to GEE Assets | ✅ Done |

---

**All assets stored at:**
projects/mironga-project-marsabit/assets/marsabit/

**Total GEE Assets (Notebook 1 + 2): 18/18** ✅

---

**Next → Notebook 3: Spatial Analysis**
- Global Moran's I on NDVI change
- LISA degradation hotspot mapping
- Distance-decay analysis around water points
- Build full Livestock Concentration Index (LCI)